In [24]:
import pandas as pd
import numpy as np
import re
import pynvml
import time


pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
arch_code = pynvml.nvmlDeviceGetArchitecture(handle)
arch_map = {
    0: "Unknown",
    2: "Kepler",
    3: "Maxwell",
    4: "Pascal",
    5: "Volta",
    6: "Turing",
    7: "Ampere",
    8: "Ada Lovelace",
    9: 'Hopper',
    10: 'Blackwell'
}

idle_power_mw, _ = pynvml.nvmlDeviceGetPowerManagementLimitConstraints(handle)
idle_power = idle_power_mw/1000

architecture = arch_map.get(arch_code, f"Unknown ({arch_code})")

path = '/home/bepi/Desktop/Ph.D_/projects/GPU_stress/code/ScalableGPUMonitoring/NCU/data/raw/ncu/gpuburnsass_1.csv'


if architecture == 'Ada Lovelace':
    # QUESTI VALORI SONO PRESI DA QUI: 
    # https://docs.nvidia.com/cuda/cuda-binary-utilities/index.html#ampere-ampere-instruction-set-table
    # E SONO SPECIFICI PER L'ARCHIRETTURA DELLA GPU (ADA)
    component_map = {
        # Floating-point / FP aritmetica
        "FP": [
            r"FADD", r"FMUL", r"FFMA", r"FMA", r"FSETP", r"FSET",
            r"FADD32I", r"FMUL32I", r"FFMA32I", r"FMA32I",
            r"FSWZADD", r"FSWZADD32I", r"FSESQ", r"FSQRT", r"FRND", r"F2F", r"F2I", r"F2IP", r"F2I.*",
            r"F2IP.*", r"F2F.*"
        ],

        # Interi / operazioni intere
        "INT": [
            r"IADD", r"IMUL", r"IMAD", r"IDP", r"IDP4A", r"IMMA", r"IMNMX", r"ISCADD",
            r"ISETP", r"ISCADD32I", r"IADD32I", r"IMUL32I", r"IMAD32I", r"I2I", r"I2IP", r"I2F",
            r"I2FP.*", r"I2IP.*"
        ],

        # ALU / operazioni logiche, spostamenti, permutazioni, movimenti
        "ALU": [
            r"MOV", r"MOV32I", r"MOVM", r"LOP", r"LOP32I", r"PRMT", r"SEL", r"SGXT",
            r"SHFL", r"SHF", r"LEA", r"LEA\.HI", r"UMOV", r"UIADD3", r"ULDC", 
            r"LOC", r"LOC32I", r"R2P", r"UR2UP", r"UR2UP.*"
        ],

        # SFU / funzioni speciali
        "SFU": [
            r"MUFU", r"RCPSQRT", r"RCP", r"RSQRT", r"SIN", r"COS", r"EX2", r"LG2",
            r"SQRT", r"LOP", r"LOP32I"  # se consideri parti “speciali”
        ],

        # Tensor / MMA / operazioni su tensor cores
        "TENSOR": [
            r"HMMA", r"HMMA\.TF32", r"HMMA32I", r"HMMA_FP16", r"DMMA", r"WMMA", r"MMAS",
            r"MMAD", r"HMNMX2", r"HMUL2", r"HMUL2_32I"
        ],

        # # Memoria condivisa / accessi in shared
        # "Shared": [
        #     r"LDS", r"STS", r"LDSM", r"X1", r"X2", r"CPY", r"COPY", r"CPY_LOCAL", # copie da global a shared
        #     r"ATOM", r"ATOMS", r"ATOMIC"  # se consideri atomiche su shared
        # ],

        # # Memoria globale (LDG, STG, global memory)
        # "GlobalMem": [
        #     r"LDG", r"STG", r"LDGSTS", r"LDGDEPBAR", r"CPY", r"CPY_GLOBAL", r"COPY_GLOBAL"
        # ],

        # # Memoria locale (LDL, STL)
        # "LocalMem": [
        #     r"LDL", r"STL", r"CPY_LOCAL", r"CPY_LCL"
        # ],

        # # Cache costanti / LDC
        # "Const_Cache": [
        #     r"LDC", r"LDCG", r"LDC.*"
        # ],

        # # Texture / cache texture
        # "Texture_Cache": [
        #     r"TEX", r"TEXL", r"TEXS", r"TEX.*"
        # ],
        # # Registri / file dei registri (accessi ai registri)
        # "REG": [
        #     # Potresti non mappare tutti qui, ma puoi lasciare come fallback
        #     r".*"  # default catch-all – ogni istruzione accede a REG
        # ],

        # # File di dati / FDS (o file di segment registers / dati)
        # "FDS": [
        #     r".*"
        # ],
    }
elif architecture == 'Ampere':
    component_map = {
        # Floating-point / FP aritmetica (FP32, FP64, varianti)
        "FP": [
            r"FADD", r"FADD32I", r"FMUL", r"FMUL32I", r"FFMA", r"FFMA32I", r"FMA", r"FMA32I",
            r"FSWZADD", r"FSWZADD32I", r"FSESQ", r"FSQRT", r"FRND",
            r"F2F", r"F2IP", r"F2I", r"F2I.*", r"F2IP.*", r"F2F.*"
        ],

        # Interi / operazioni intere (arithmetic integer, multiply, dp, etc.)
        "INT": [
            r"IADD", r"IADD32I", r"IMUL", r"IMUL32I", r"IMAD", r"IMAD32I",
            r"IDP", r"IDP4A", r"IMMA", r"IMNMX", r"ISCADD", r"ISCADD32I",
            r"ISETP", r"I2I", r"I2IP", r"I2F", r"I2FP.*", r"I2IP.*"
        ],

        # ALU / operazioni logiche, spostamenti, permutazioni, movimenti
        "ALU": [
            r"MOV", r"MOV32I", r"MOVM",
            r"LOP", r"LOP32I", r"PRMT", r"SEL", r"SGXT",
            r"SHFL", r"SHF", r"LEA", r"LEA\.HI", r"UMOV", r"UIADD3", r"ULDC",
            r"LOC", r"LOC32I", r"R2P", r"UR2UP", r"UR2UP.*"
        ],

        # SFU / funzioni speciali (trigonometria, reciprocali, etc.)
        "SFU": [
            r"MUFU", r"RCP", r"RSQRT", r"RCPSQRT", r"SIN", r"COS", r"EX2", r"LG2", r"SQRT"
        ],

        # Tensor / MMA / operazioni su tensor cores
        "TENSOR": [
            r"HMMA", r"HMMA\.TF32", r"HMMA32I", r"HMMA_FP16", r"DMMA", r"WMMA",
            r"MMAS", r"MMAD", r"HMNMX2", r"HMUL2", r"HMUL2_32I"
        ],

        # # Memoria condivisa / accessi in shared
        # "Shared": [
        #     r"LDS", r"STS", r"LDSM", r"X1", r"X2",
        #     r"CPY", r"COPY", r"CPY_LOCAL",
        #     r"ATOM", r"ATOMS", r"ATOMIC"
        # ],

        # # Memoria globale (LDG, STG, global memory)
        # "GlobalMem": [
        #     r"LDG", r"STG", r"LDGSTS", r"LDGDEPBAR", r"CPY", r"CPY_GLOBAL", r"COPY_GLOBAL"
        # ],

        # # Memoria locale (LDL, STL)
        # "LocalMem": [
        #     r"LDL", r"STL", r"CPY_LOCAL", r"CPY_LCL"
        # ],

        # # Cache costanti / LDC
        # "Const_Cache": [
        #     r"LDC", r"LDCG", r"LDC.*"
        # ],

        # # Texture / cache texture
        # "Texture_Cache": [
        #     r"TEX", r"TEXL", r"TEXS", r"TEX.*"
        # ],

        # # Registri / file dei registri (accessi ai registri)
        # "REG": [
        #     r".*"   # catch-all: ogni istruzione accede almeno a file registro
        # ],

        # # File di dati / FDS (o file segment registers / dati)
        # "FDS": [
        #     r".*"
        # ],
    }

max_power = {
    "FP": 0.2,
    "INT": 0.25,
    "ALU": 0.2,
    "SFU": 0.5,
    # "Shared": 1.0,
    # "GlobalMem": 52.0,
    # "LocalMem": 52.0,
    # "Const_Cache": 0.4,
    # "Texture_Cache": 0.9,
    # "REG": 0.3,
    # "FDS": 0.5,
    "Const_SM": 0.813,
}

df = pd.read_csv(f"{path}", header=None, names=["Address", "Instruction"])

df["Instruction"] = df["Instruction"].str.strip()

df["Opcode"] = df["Instruction"].str.split().str[0]

opcode_counts = df["Opcode"].value_counts().reset_index()

# print(opcode_counts)
opcode_counts = opcode_counts[~opcode_counts["Opcode"].str.lower().isin(["source", "compare", "ampere_sgemm_64x64_nn"])]

counts = {k: 0 for k in max_power}
count = 0
for _, row in opcode_counts.iterrows():
    opcode = row["Opcode"]
    count = row["count"]
    for comp, patterns in component_map.items():
        if any(re.match(p, opcode) for p in patterns):
            count += 1
            counts[comp] += count
            break
    # tutte le istruzioni accedono a REG e FDS
    # counts["REG"] += count
    # counts["FDS"] += count

spec_linear_components = [
    "FP", "REG", "INT", "FDS", "Texture_Cache",
    "Const_Cache", "GlobalMem", "LocalMem"
]

def piecewise_linear(rate):
    if rate <= 0:
        return 0
    return 0.1365 * np.log(rate) + 1.001375
    
total_insts = opcode_counts["count"].sum()
access_rates = {k: v / total_insts for k, v in counts.items()}

adj_access_rates = {}
for k, v in access_rates.items():
    if k in spec_linear_components:
        adj_access_rates[k] = piecewise_linear(v)
    else:
        adj_access_rates[k] = v

runtime_power_base = sum(max_power[k] * access_rates[k] for k in max_power)

# runtime_power_base = sum(max_power[k] * adj_access_rates[k] for k in max_power)

Num_SMs = 24
alpha = (10 - 1.1) / Num_SMs
beta = 1.1

active_sms = np.arange(1, Num_SMs + 1)
# ATTENZIONE: QUESTO MODELLO ASSUME CHE LA POTENZA SIA IMPIEGATA SOLO DA SM CHE SONO COMPLETAMENTE ATTIVI: QUESTO DETERMINA A DIFFERENZA W.R.T LA POWER REALE
# IL POWER GATING È IL FENOMENO CHE GENERA QUESTO DELTA W.R.T. LA POWER REALE
runtime_power = runtime_power_base * np.log10(alpha * active_sms + beta)

tot_power = idle_power + (runtime_power[-1]*Num_SMs)
print(tot_power)


8.253938763376933


In [16]:
adj_access_rates

{'FP': np.float64(0.8881967531347261),
 'INT': np.float64(0.6704700503055081),
 'ALU': np.float64(0.13079667063020214),
 'SFU': np.float64(0.0),
 'Shared': np.float64(0.0558858501783591),
 'Const_Cache': 0,
 'Texture_Cache': 0,
 'REG': np.float64(1.0365322417531773),
 'FDS': np.float64(1.0017447610662293),
 'Const_SM': np.float64(0.0)}

In [ ]:
import pandas as pd
import numpy as np
import re

path = '/home/bepi/Desktop/Ph.D_/projects/GPU_stress/code/ScalableGPUMonitoring/NCU/data/raw/ncu/gpuburnsass_1.csv'

# QUESTI VALORI SONO PRESI DA QUI: 
component_map = {
    # Floating-point / FP aritmetica
    "FP": [
        r"FADD", r"FMUL", r"FFMA", r"FMA", r"FSETP", r"FSET",
        r"FADD32I", r"FMUL32I", r"FFMA32I", r"FMA32I",
        r"FSWZADD", r"FSWZADD32I", r"FSESQ", r"FSQRT", r"FRND", r"F2F", r"F2I", r"F2IP", r"F2I.*",
        r"F2IP.*", r"F2F.*"
    ],

    # Interi / operazioni intere
    "INT": [
        r"IADD", r"IMUL", r"IMAD", r"IDP", r"IDP4A", r"IMMA", r"IMNMX", r"ISCADD",
        r"ISETP", r"ISCADD32I", r"IADD32I", r"IMUL32I", r"IMAD32I", r"I2I", r"I2IP", r"I2F",
        r"I2FP.*", r"I2IP.*"
    ],

    # ALU / operazioni logiche, spostamenti, permutazioni, movimenti
    "ALU": [
        r"MOV", r"MOV32I", r"MOVM", r"LOP", r"LOP32I", r"PRMT", r"SEL", r"SGXT",
        r"SHFL", r"SHF", r"LEA", r"LEA\.HI", r"UMOV", r"UIADD3", r"ULDC", 
        r"LOC", r"LOC32I", r"R2P", r"UR2UP", r"UR2UP.*"
    ],

    # SFU / funzioni speciali
    "SFU": [
        r"MUFU", r"RCPSQRT", r"RCP", r"RSQRT", r"SIN", r"COS", r"EX2", r"LG2",
        r"SQRT", r"LOP", r"LOP32I"  # se consideri parti “speciali”
    ],

    # Tensor / MMA / operazioni su tensor cores
    "TENSOR": [
        r"HMMA", r"HMMA\.TF32", r"HMMA32I", r"HMMA_FP16", r"DMMA", r"WMMA", r"MMAS",
        r"MMAD", r"HMNMX2", r"HMUL2", r"HMUL2_32I"
    ],

    # Memoria condivisa / accessi in shared
    "Shared": [
        r"LDS", r"STS", r"LDSM", r"X1", r"X2", r"CPY", r"COPY", r"CPY_LOCAL", # copie da global a shared
        r"ATOM", r"ATOMS", r"ATOMIC"  # se consideri atomiche su shared
    ],

    # Memoria globale (LDG, STG, global memory)
    "GlobalMem": [
        r"LDG", r"STG", r"LDGSTS", r"LDGDEPBAR", r"CPY", r"CPY_GLOBAL", r"COPY_GLOBAL"
    ],

    # Memoria locale (LDL, STL)
    "LocalMem": [
        r"LDL", r"STL", r"CPY_LOCAL", r"CPY_LCL"
    ],

    # Cache costanti / LDC
    "Const_Cache": [
        r"LDC", r"LDCG", r"LDC.*"
    ],

    # Texture / cache texture
    "Texture_Cache": [
        r"TEX", r"TEXL", r"TEXS", r"TEX.*"
    ],
    # Registri / file dei registri (accessi ai registri)
    "REG": [
        # Potresti non mappare tutti qui, ma puoi lasciare come fallback
        r".*"  # default catch-all – ogni istruzione accede a REG
    ],

    # File di dati / FDS (o file di segment registers / dati)
    "FDS": [
        r".*"
    ],
}

max_power = {
    "FP": 0.2,
    "INT": 0.25,
    "ALU": 0.2,
    "SFU": 0.5,
    "Shared": 1.0,
    "GlobalMem": 52.0,
    "LocalMem": 52.0,
    "Const_Cache": 0.4,
    "Texture_Cache": 0.9,
    "REG": 0.3,
    "FDS": 0.5,
    "Const_SM": 0.813,
}

df = pd.read_csv(f"{path}", header=None, names=["Address", "Instruction"])

df["Instruction"] = df["Instruction"].str.strip()

df["Opcode"] = df["Instruction"].str.split().str[0]

opcode_counts = df["Opcode"].value_counts().reset_index()

# print(opcode_counts)
opcode_counts = opcode_counts[~opcode_counts["Opcode"].str.lower().isin(["source", "compare", "ampere_sgemm_64x64_nn"])]

counts = {k: 0 for k in max_power}
count = 0
for _, row in opcode_counts.iterrows():
    opcode = row["Opcode"]
    count = row["count"]
    for comp, patterns in component_map.items():
        if any(re.match(p, opcode) for p in patterns):
            count += 1
            counts[comp] += count
            break
    # tutte le istruzioni accedono a REG e FDS
    counts["REG"] += count
    counts["FDS"] += count
# print(f'count: {count}')
# print(f'expected count: {opcode_counts}')
# === 4. Calcola runtime power medio per 30 SM ===
total_insts = opcode_counts["count"].sum()
access_rates = {k: v / total_insts for k, v in counts.items()}
runtime_power_base = sum(max_power[k] * access_rates[k] for k in max_power)

print(f"\nEstimated base runtime power (normalized): {runtime_power_base:.4f}\n")

Num_SMs = 86
alpha = (10 - 1.1) / Num_SMs
beta = 1.1

active_sms = np.arange(1, Num_SMs + 1)
# ATTENZIONE: QUESTO MODELLO ASSUME CHE LA POTENZA SIA IMPIEGATA SOLO DA SM CHE SONO COMPLETAMENTE ATTIVI: QUESTO DETERMINA A DIFFERENZA W.R.T LA POWER REALE
# IL POWER GATING È IL FENOMENO CHE GENERA QUESTO DELTA W.R.T. LA POWER REALE
runtime_power = runtime_power_base * np.log10(alpha * active_sms + beta)

idle_power = 5.0 
gpu_power = runtime_power + idle_power
print(sum(gpu_power)/60)

print(f"\nEstimated GPU power (W) vs active SMs:\n{gpu_power}")

memory_insts = counts["GlobalMem"] + counts["LocalMem"]
non_memory_insts = sum(counts[k] for k in counts if k not in ["GlobalMem", "LocalMem"])
mem_access_intensity = memory_insts / non_memory_insts if non_memory_insts > 0 else 0

rho = 21.505
lambda_ = 5.5
mu = 0.120

max_temp = (rho * runtime_power_base) + (lambda_ * mem_access_intensity) + mu

print(f'Maximum temperature: {max_temp}')
# print(df["Opcode"].unique())


Estimated base runtime power (normalized): 1.2077

8.360647819931138

Estimated GPU power (W) vs active SMs:
[5.09715335 5.140422   5.18039172 5.21753012 5.25221192 5.28474205
 5.31537196 5.34431146 5.3717374  5.39780028 5.42262918 5.4463357
 5.46901692 5.49075789 5.51163345 5.53170988 5.55104611 5.56969478
 5.58770312 5.60511365 5.62196477 5.63829134 5.65412501 5.66949468
 5.68442677 5.69894551 5.71307318 5.72683027 5.74023575 5.75330714
 5.76606068 5.77851147 5.79067354 5.80255999 5.81418303 5.82555409
 5.83668385 5.84758235 5.858259   5.86872265 5.87898164 5.88904381
 5.89891659 5.90860696 5.91812154 5.92746661 5.93664809 5.94567161
 5.95454252 5.96326589 5.97184654 5.98028909 5.98859789 5.99677713
 6.00483077 6.01276262 6.02057632 6.02827531 6.03586294 6.04334236
 6.05071663 6.05798866 6.06516124 6.07223706 6.0792187  6.08610863
 6.09290923 6.09962278 6.10625148 6.11279746 6.11926275 6.12564932
 6.13195906 6.13819379 6.14435529 6.15044524 6.1564653  6.16241705
 6.16830202 6.174121